In [1]:
import json
import os

import gc
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from torch import GradScaler
from torch.cuda.amp import autocast
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from internal.data.mil_dataset import MILDatasetMemmapRanges
from internal.data.shuffle_and_cap_bag import ShuffleAndCapBag
from internal.nn.attention_mil import AttentionMIL

cuda_is_available = True
device = torch.device("cuda")

config = json.load(
    open(os.path.join("processed", "config.json"), "r")
)
OUT_DIR = config["OUT_DIR"]
X_PATH = config["X_PATH"]
M_PATH = config["M_PATH"]
Y_PATH = config["Y_PATH"]
IDX_PATH = config["IDX_PATH"]
LOG_PATH = config["LOG_PATH"]
PATCH_SIZE = config["PATCH_SIZE"]
N_PATCHES = config["N_PATCHES"]
MARGIN = config["MARGIN"]
MASK_PATCH_FRAC = config["MASK_PATCH_FRAC"]
MIN_MASK_PIXELS_SLIDE = config["MIN_MASK_PIXELS_SLIDE"]
MIN_MASK_IN_PATCH = config["MIN_MASK_IN_PATCH"]
MIN_TISSUE_FRAC = config["MIN_TISSUE_FRAC"]
MIN_PATCH_PER_SLIDE = config["MIN_PATCH_PER_SLIDE"]
MIN_CENTER_DIST = config["MIN_CENTER_DIST"]
MAX_TRIES_PER_SLIDE = config["MAX_TRIES_PER_SLIDE"]
MAX_TRIES_PER_PATCH = config["MAX_TRIES_PER_PATCH"]
CLASS2ID = config["CLASS2ID"]
ID2CLASS = config["ID2CLASS"]

RETURN_META = True

In [2]:
def run_epoch(model, loader, optimizer=None, scaler=None, device="cuda"):
    train = optimizer is not None
    model.train(train)

    all_pred, all_true = [], []
    total_loss, n = 0.0, 0
    loss_fn = nn.CrossEntropyLoss()

    # choose grad context
    grad_ctx = torch.enable_grad if train else torch.inference_mode

    for batch in tqdm(loader, desc="Train" if train else "Val", unit="batch"):
        xcat, y, bag_sizes = batch[:3]
        xcat = xcat.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        bag_sizes = bag_sizes.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with grad_ctx():
            with autocast(enabled=(scaler is not None) and train):
                logits = model(xcat, bag_sizes)
                loss = loss_fn(logits, y)

            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += float(loss.detach().cpu())
        n += 1

        pred = logits.argmax(dim=1).detach().cpu().tolist()
        all_pred.extend(pred)
        all_true.extend(y.detach().cpu().tolist())

    macro_f1 = f1_score(all_true, all_pred, average="macro")
    return total_loss / max(n, 1), macro_f1

In [3]:
def mil_collate(batch):
    # batch items can be (x,y) or (x,y,meta)
    if len(batch[0]) == 3:
        xs, ys, metas = zip(*batch)
    else:
        xs, ys = zip(*batch)
        metas = None

    ys = torch.tensor(ys, dtype=torch.long)
    bag_sizes = torch.tensor([x.size(0) for x in xs], dtype=torch.long)
    xcat = torch.cat(xs, dim=0)  # (sum n_i, C, H, W)

    if metas is None:
        return xcat, ys, bag_sizes
    return xcat, ys, bag_sizes, metas


def mil_collate_with_meta(batch):
    xs = [b[0] for b in batch]
    ys = torch.stack([b[1] if torch.is_tensor(b[1]) else torch.tensor(b[1], dtype=torch.long) for b in batch]).long()
    metas = [b[2] for b in batch]  # list of dicts

    bag_sizes = torch.tensor([x.size(0) for x in xs], dtype=torch.long)
    xcat = torch.cat(xs, dim=0)

    return xcat, ys, bag_sizes, metas

def mil_collate_concat(batch):
    if RETURN_META:
        xs, ys, _ = zip(*batch)   # each x: (n_i,C,H,W)
    else:
        xs, ys = zip(*batch)      # each x: (n_i,C,H,W)
    bag_sizes = torch.tensor([x.shape[0] for x in xs], dtype=torch.long)
    x = torch.cat(xs, dim=0)  # (sum n_i, C,H,W)
    y = torch.stack(ys)       # (B,)
    return x, y, bag_sizes

In [4]:
train_ds = MILDatasetMemmapRanges(
    x_path=X_PATH,
    y_path=Y_PATH,
    idx_path=IDX_PATH,
    m_path=M_PATH,
    patch_size=PATCH_SIZE,
    bag_transform=ShuffleAndCapBag(max_instances=16),
    return_meta=True
)
val_ds   = MILDatasetMemmapRanges(
    x_path=X_PATH,
    y_path=Y_PATH,
    idx_path=IDX_PATH,
    m_path=M_PATH,
    patch_size=PATCH_SIZE,
    bag_transform=ShuffleAndCapBag(max_instances=16),
    return_meta=True
)

train_loader = DataLoader(
    train_ds,
    batch_size=2,              # number of slides per batch
    shuffle=True,
    # num_workers=0,
    pin_memory=cuda_is_available,
    collate_fn=mil_collate_concat,   # or mil_collate_list
)
val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    # num_workers=0,
    pin_memory=False,
    collate_fn=mil_collate,
)

In [5]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

model = AttentionMIL(n_classes=4).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scaler = GradScaler(enabled=(device=="cuda"))

best_f1 = -1
for epoch in range(1, 21):
    print(f"Epoch {epoch:02d} -------------------------------")
    tr_loss, tr_f1 = run_epoch(model, train_loader, optimizer=optimizer, scaler=scaler, device=device)
    gc.collect()
    if cuda_is_available:
        torch.cuda.empty_cache()
    va_loss, va_f1 = run_epoch(model, val_loader, optimizer=None, scaler=None, device=device)

    print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} f1 {tr_f1:.4f} | val loss {va_loss:.4f} f1 {va_f1:.4f}")

    if va_f1 > best_f1:
        best_f1 = va_f1
        torch.save({"model": model.state_dict()}, "best_mil.pt")


Epoch 01 -------------------------------


Train:   0%|          | 0/291 [00:00<?, ?batch/s]

/home/andre/university/AN2DL-Challenge-2/notebooks-v2/internal/data/mil_dataset.py:56: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  x = torch.from_numpy(x_np).permute(0, 3, 1, 2)  # uint8 tensor (non-writable is fine)
/tmp/ipykernel_6188/3901953080.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(scaler is not None) and train):


Val:   0%|          | 0/581 [00:00<?, ?batch/s]

/tmp/ipykernel_6188/3901953080.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(scaler is not None) and train):


...

/tmp/ipykernel_6188/3901953080.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(scaler is not None) and train):


Epoch 08 | train loss 0.5978 f1 0.7219 | val loss 1.8019 f1 0.3358
Epoch 09 -------------------------------


Train:   0%|          | 0/291 [00:00<?, ?batch/s]

/tmp/ipykernel_6188/3901953080.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(scaler is not None) and train):


KeyboardInterrupt: 